# 01 · Build workshop data (facilitator)

**Facilitator notebook.** Run once before attendees start (or have them watch you run it).

Creates:
1. Synthetic claims tables (~700 claims, planted duplicates, PH0013 sibling trap)
2. PK / FK constraints so Genie can pick up joins
3. Twelve pet-claim PDFs on the `claim_docs` volume (Sparky / Pine Vet planted pair)
4. The four `CLM-PET-*` rows inside `claims`, so the PDF duplicate was also **Paid**

`random.seed(42)` keeps counts stable. Duplicate definition: same `pet_id + clinic_id + visit_date`.


In [ ]:
%run ./00_setup


In [ ]:
from workshop_data import generate

tables = generate()
for name, rows in tables.items():
    spark.createDataFrame(rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, len(rows))


## Primary / foreign keys (Genie uses these as joins)


In [ ]:
# UC requires PK/FK columns to be NOT NULL before ADD CONSTRAINT.
# Drop → SET NOT NULL → add PK → add FK. Genie reads these as suggested joins.
stmts = [
    # Drop existing (idempotent re-runs)
    "ALTER TABLE policyholders DROP CONSTRAINT IF EXISTS pk_policyholders",
    "ALTER TABLE pets DROP CONSTRAINT IF EXISTS pk_pets",
    "ALTER TABLE policies DROP CONSTRAINT IF EXISTS pk_policies",
    "ALTER TABLE vet_clinics DROP CONSTRAINT IF EXISTS pk_vet_clinics",
    "ALTER TABLE claims DROP CONSTRAINT IF EXISTS pk_claims",
    "ALTER TABLE claim_payments DROP CONSTRAINT IF EXISTS pk_claim_payments",
    "ALTER TABLE pets DROP CONSTRAINT IF EXISTS fk_pets_policyholder",
    "ALTER TABLE policies DROP CONSTRAINT IF EXISTS fk_policies_pet",
    "ALTER TABLE policies DROP CONSTRAINT IF EXISTS fk_policies_policyholder",
    "ALTER TABLE claims DROP CONSTRAINT IF EXISTS fk_claims_policy",
    "ALTER TABLE claims DROP CONSTRAINT IF EXISTS fk_claims_pet",
    "ALTER TABLE claims DROP CONSTRAINT IF EXISTS fk_claims_clinic",
    "ALTER TABLE claim_payments DROP CONSTRAINT IF EXISTS fk_payments_claim",
    # PK columns NOT NULL
    "ALTER TABLE policyholders ALTER COLUMN policyholder_id SET NOT NULL",
    "ALTER TABLE pets ALTER COLUMN pet_id SET NOT NULL",
    "ALTER TABLE policies ALTER COLUMN policy_id SET NOT NULL",
    "ALTER TABLE vet_clinics ALTER COLUMN clinic_id SET NOT NULL",
    "ALTER TABLE claims ALTER COLUMN claim_id SET NOT NULL",
    "ALTER TABLE claim_payments ALTER COLUMN payment_id SET NOT NULL",
    # FK columns NOT NULL
    "ALTER TABLE pets ALTER COLUMN policyholder_id SET NOT NULL",
    "ALTER TABLE policies ALTER COLUMN pet_id SET NOT NULL",
    "ALTER TABLE policies ALTER COLUMN policyholder_id SET NOT NULL",
    "ALTER TABLE claims ALTER COLUMN policy_id SET NOT NULL",
    "ALTER TABLE claims ALTER COLUMN pet_id SET NOT NULL",
    "ALTER TABLE claims ALTER COLUMN clinic_id SET NOT NULL",
    "ALTER TABLE claim_payments ALTER COLUMN claim_id SET NOT NULL",
    # Primary keys
    "ALTER TABLE policyholders ADD CONSTRAINT pk_policyholders PRIMARY KEY (policyholder_id)",
    "ALTER TABLE pets ADD CONSTRAINT pk_pets PRIMARY KEY (pet_id)",
    "ALTER TABLE policies ADD CONSTRAINT pk_policies PRIMARY KEY (policy_id)",
    "ALTER TABLE vet_clinics ADD CONSTRAINT pk_vet_clinics PRIMARY KEY (clinic_id)",
    "ALTER TABLE claims ADD CONSTRAINT pk_claims PRIMARY KEY (claim_id)",
    "ALTER TABLE claim_payments ADD CONSTRAINT pk_claim_payments PRIMARY KEY (payment_id)",
    # Foreign keys
    "ALTER TABLE pets ADD CONSTRAINT fk_pets_policyholder FOREIGN KEY (policyholder_id) REFERENCES policyholders(policyholder_id)",
    "ALTER TABLE policies ADD CONSTRAINT fk_policies_pet FOREIGN KEY (pet_id) REFERENCES pets(pet_id)",
    "ALTER TABLE policies ADD CONSTRAINT fk_policies_policyholder FOREIGN KEY (policyholder_id) REFERENCES policyholders(policyholder_id)",
    "ALTER TABLE claims ADD CONSTRAINT fk_claims_policy FOREIGN KEY (policy_id) REFERENCES policies(policy_id)",
    "ALTER TABLE claims ADD CONSTRAINT fk_claims_pet FOREIGN KEY (pet_id) REFERENCES pets(pet_id)",
    "ALTER TABLE claims ADD CONSTRAINT fk_claims_clinic FOREIGN KEY (clinic_id) REFERENCES vet_clinics(clinic_id)",
    "ALTER TABLE claim_payments ADD CONSTRAINT fk_payments_claim FOREIGN KEY (claim_id) REFERENCES claims(claim_id)",
]
for stmt in stmts:
    try:
        spark.sql(stmt)
        print("ok:", stmt)
    except Exception as e:
        print("skip:", stmt, "→", e)
print("PK/FK applied.")


## Table comments


In [ ]:
comments = [
    "COMMENT ON TABLE claims IS 'One row per submitted claim. Contains some duplicate submissions for the same pet, clinic, and visit date.'",
    "ALTER TABLE claims ALTER COLUMN claim_id COMMENT 'Unique claim ID. A resubmission of the same visit gets a NEW claim_id.'",
    "ALTER TABLE claims ALTER COLUMN invoice_number COMMENT 'Clinic invoice number as submitted; resubmissions may reuse or alter it. Do not treat as a unique key.'",
    "ALTER TABLE claims ALTER COLUMN pet_id COMMENT 'The pet treated on this visit. FK to pets.pet_id.'",
    "ALTER TABLE claims ALTER COLUMN denial_reason COMMENT 'Populated when claim_status = Denied. Use Duplicate claim for caught resubmissions.'",
    "COMMENT ON TABLE claim_payments IS 'Payouts issued for claims with status Paid. Join to claims on claim_id.'",
    "COMMENT ON TABLE policies IS 'One row per pet insurance policy. reimbursement_rate is a decimal (0.80 = 80%).'",
    "COMMENT ON TABLE pets IS 'Insured pets. Multiple pets can share a policyholder_id (household).'",
    "COMMENT ON TABLE policyholders IS 'Customers. policyholder_id is the household key.'",
    "COMMENT ON TABLE vet_clinics IS 'Clinics that submitted claims.'",
    "COMMENT ON TABLE earned_premium_monthly IS 'Earned premium by policy and month. Use with claim_payments for loss ratio.'",
    "COMMENT ON TABLE investment_holdings IS 'Synthetic portfolio snapshot as of 2026-09-30. Stretch content only.'",
]
for stmt in comments:
    spark.sql(stmt)
print("Comments applied.")


## Answer key + duplicate view


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW v_suspected_duplicate_claims AS
WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank,
    FIRST_VALUE(claim_id) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_claim_id
  FROM claims c
)
SELECT * FROM ranked WHERE visit_rank > 1
""")

print("=== Facilitator answer key ===")
display(spark.sql("""
SELECT
  COUNT(*) AS suspected_duplicates,
  COUNT_IF(d.claim_status = 'Paid') AS duplicates_paid,
  COUNT_IF(d.claim_status = 'Denied' AND d.denial_reason = 'Duplicate claim') AS duplicates_caught,
  ROUND(SUM(p.payment_amount), 2) AS leakage_usd
FROM v_suspected_duplicate_claims d
LEFT JOIN claim_payments p USING (claim_id)
"""))

print("Planted PDF claims in claims table:")
display(spark.sql("""
SELECT claim_id, pet_id, clinic_id, visit_date, invoice_number, claim_status, claimed_amount
FROM claims
WHERE claim_id LIKE 'CLM-PET-%'
ORDER BY claim_id
"""))

print("PH0013 trap check (sibling pets same day — should NOT be duplicates):")
display(spark.sql("""
SELECT c.claim_id, c.pet_id, p.policyholder_id, c.clinic_id, c.visit_date, c.claim_status
FROM claims c
JOIN pets p USING (pet_id)
WHERE p.policyholder_id = 'PH0013'
ORDER BY c.visit_date, c.pet_id
"""))
print("Real duplicates for PH0013 (expect 0 rows):")
display(spark.sql("""
SELECT d.*
FROM v_suspected_duplicate_claims d
JOIN pets p USING (pet_id)
WHERE p.policyholder_id = 'PH0013'
"""))


## Generate claim PDFs → volume


In [ ]:
from pathlib import Path
from generate_pet_claim_pdfs import generate

dest = Path(PDF_DIR)
written = generate(dest)
print(f"Wrote {len(written)} PDFs under {dest}")
for p in written:
    print(" ", p.relative_to(dest) if dest in p.parents or str(p).startswith(str(dest)) else p)

display(spark.sql(f"""
SELECT path, length, modificationTime
FROM read_files('{PDF_DIR}', format => 'binaryFile')
WHERE path LIKE '%.pdf'
ORDER BY path
"""))
